# Batch ABI benchmark

`propaq_noise_damping_batch` / `propaq_truncator_keep_batch` are optional.
When a plugin exports them, propaq calls one batch entry point per parallel
chunk of terms instead of calling the scalar entry point once per term,
amortizing the FFI boundary cost across the chunk.

This notebook measures the performance difference between batch and scalar entry points.

## Building scalar-only vs scalar+batch variants

In [1]:
import subprocess
from pathlib import Path

BUILD_DIR = Path("_build").resolve()
BUILD_DIR.mkdir(exist_ok=True)

SCALAR_PLUS_BATCH_SRC = Path("../c/truncation/pareto_truncator.c").resolve().read_text()

# Same ctx/parsing/keep logic, but the batch entry point is deleted --
# everything up to (not including) `propaq_truncator_keep_batch` is kept.
cut = SCALAR_PLUS_BATCH_SRC.index("int32_t propaq_truncator_keep_batch")
SCALAR_ONLY_SRC = SCALAR_PLUS_BATCH_SRC[:cut]

(BUILD_DIR / "pareto_scalar_only.c").write_text(SCALAR_ONLY_SRC)
(BUILD_DIR / "pareto_scalar_batch.c").write_text(SCALAR_PLUS_BATCH_SRC)

for name in ["pareto_scalar_only", "pareto_scalar_batch"]:
    subprocess.run(
        ["gcc", "-shared", "-fPIC", "-O2", "-o", str(BUILD_DIR / f"{name}.so"), str(BUILD_DIR / f"{name}.c"), "-lm"],
        check=True,
    )

# Confirm the scalar-only build genuinely has no batch symbol, and the other does.
nm_only = subprocess.run(["nm", "-D", str(BUILD_DIR / "pareto_scalar_only.so")], capture_output=True, text=True).stdout
nm_batch = subprocess.run(["nm", "-D", str(BUILD_DIR / "pareto_scalar_batch.so")], capture_output=True, text=True).stdout
print("scalar-only exports keep_batch:", "keep_batch" in nm_only)
print("scalar+batch exports keep_batch:", "keep_batch" in nm_batch)

scalar-only exports keep_batch: False
scalar+batch exports keep_batch: True


## Isolated raw-FFI benchmark

Call `propaq_truncator_keep` in a
plain Python loop `N` times, versus calling `propaq_truncator_keep_batch`
once on the same `N` elements, via `ctypes` directly.

In [2]:
import ctypes
import random
import time

lib = ctypes.CDLL(str(BUILD_DIR / "pareto_scalar_batch.so"))
lib.propaq_truncator_create.restype = ctypes.c_void_p
lib.propaq_truncator_create.argtypes = [ctypes.c_char_p]
lib.propaq_truncator_keep.restype = ctypes.c_int32
lib.propaq_truncator_keep.argtypes = [ctypes.c_void_p, ctypes.c_uint32, ctypes.c_double, ctypes.c_uint32]
lib.propaq_truncator_keep_batch.restype = ctypes.c_int32
lib.propaq_truncator_keep_batch.argtypes = [
    ctypes.c_void_p, ctypes.POINTER(ctypes.c_uint32), ctypes.POINTER(ctypes.c_double),
    ctypes.POINTER(ctypes.c_uint32), ctypes.POINTER(ctypes.c_uint8), ctypes.c_size_t,
]

ctx = lib.propaq_truncator_create(b'{"threshold": 1e-6, "alpha": 0.05}')

random.seed(1)
N = 200_000
weights = (ctypes.c_uint32 * N)(*(random.randint(0, 10) for _ in range(N)))
coeffs = (ctypes.c_double * N)(*(random.uniform(1e-8, 1.0) for _ in range(N)))
active = (ctypes.c_uint32 * N)(*([0] * N))
out = (ctypes.c_uint8 * N)()

t0 = time.perf_counter()
for i in range(N):
    lib.propaq_truncator_keep(ctx, weights[i], coeffs[i], 0)
t1 = time.perf_counter()
lib.propaq_truncator_keep_batch(ctx, weights, coeffs, active, out, N)
t2 = time.perf_counter()

scalar_ffi_ms = (t1 - t0) * 1000
batch_ffi_ms = (t2 - t1) * 1000
print(f"N={N} keep decisions")
print(f"scalar path ({N} FFI calls):   {scalar_ffi_ms:.2f} ms")
print(f"batch path  (1 FFI call):      {batch_ffi_ms:.2f} ms")
print(f"speedup: {scalar_ffi_ms / batch_ffi_ms:.1f}x")

N=200000 keep decisions
scalar path (200000 FFI calls):   382.17 ms
batch path  (1 FFI call):      1.76 ms
speedup: 217.6x


## Does that show up in a real propagation?

In [3]:
import random

from propaq._rust_core import PauliString
from propaq.circuits import PauliCircuit, PauliRotation
from propaq.datatypes import PauliTermSum
from propaq.propagators import PauliPropagator
from propaq.truncation import NativeTruncator

N_QUBITS = 6

random.seed(0)

def random_circuit(depth=150):
    rotations = []
    for _ in range(depth):
        x = random.randint(0, 2**N_QUBITS - 1)
        z = random.randint(0, 2**N_QUBITS - 1)
        if x == 0 and z == 0:
            x = 1
        rotations.append(PauliRotation(PauliString(x, z, N_QUBITS), random.uniform(0.05, 0.6)))
    return PauliCircuit(rotations)

def observable():
    ts = PauliTermSum()
    ts.add(PauliString(0, 1, N_QUBITS), 1.0)
    return ts

CIRCUIT = random_circuit()
OBSERVABLE = observable()
CFG = '{"threshold": 1e-6, "alpha": 0.05}'

def run(so_path, n_threads=8):
    trunc = NativeTruncator(so_path, config=CFG)
    prop = PauliPropagator(truncation=trunc, n_threads=n_threads)
    result = prop.expectation_value(OBSERVABLE, CIRCUIT, initial_state=0)
    return result.expectation_value, max(result.n_terms)

val_scalar, peak_terms_scalar = run(str(BUILD_DIR / "pareto_scalar_only.so"))
val_batch, peak_terms_batch = run(str(BUILD_DIR / "pareto_scalar_batch.so"))
print(f"scalar-only result: {val_scalar!r}  (peak live terms: {peak_terms_scalar})")
print(f"scalar+batch result: {val_batch!r}  (peak live terms: {peak_terms_batch})")

scalar-only result: 0.11827315004523578  (peak live terms: 4095)
scalar+batch result: 0.11827315004523578  (peak live terms: 4095)


In [4]:
import timeit

N_REPEATS = 5

scalar_time = timeit.timeit(lambda: run(str(BUILD_DIR / "pareto_scalar_only.so")), number=N_REPEATS) / N_REPEATS
batch_time = timeit.timeit(lambda: run(str(BUILD_DIR / "pareto_scalar_batch.so")), number=N_REPEATS) / N_REPEATS

print(f"scalar-only:   {scalar_time * 1000:.2f} ms/run")
print(f"scalar+batch:  {batch_time * 1000:.2f} ms/run")
print(f"end-to-end speedup: {scalar_time / batch_time:.2f}x")

scalar-only:   1553.12 ms/run
scalar+batch:  1056.09 ms/run
end-to-end speedup: 1.47x
